# Step 3 — Fine-Tuning SQLCoder 7B

**Important:** Ollama does not support fine-tuning directly.
We fine-tune using HuggingFace + LoRA, save the adapter, then re-register it with Ollama.

Flow:
1. Load sqlcoder-7b-2 weights from HuggingFace (same weights Ollama uses)
2. Apply QLoRA and train on your 80k rows
3. Save adapter → merge → export to Ollama as `sqlcoder-finetuned`

**M4 16GB settings:** batch=1, grad_accum=16, fp16, max_seq=512

In [ ]:
import sys
!{sys.executable} -m pip install -q transformers accelerate peft datasets trl
print('✅ Done')

In [ ]:
import os, json, csv, time, re
from pathlib import Path
import torch

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

DEVICE      = 'mps' if torch.backends.mps.is_available() else 'cpu'
MODEL_NAME  = 'defog/sqlcoder-7b-2'
MAX_SEQ_LEN = 512
OUTPUT_DIR  = Path('checkpoints')
OUTPUT_DIR.mkdir(exist_ok=True)
Path('logs').mkdir(exist_ok=True)

print(f'Device  : {DEVICE.upper()}')
print(f'PyTorch : {torch.__version__}')

In [ ]:
from transformers import AutoTokenizer

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f'✅ Tokenizer ready — vocab: {tokenizer.vocab_size:,}')

In [ ]:
from transformers import AutoModelForCausalLM

print(f'Loading model → {DEVICE.upper()} (float16) ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={'': DEVICE},
    trust_remote_code=True,
)
model.config.use_cache      = False
model.config.pretraining_tp = 1

total = sum(p.numel() for p in model.parameters())
print(f'Parameters : {total/1e9:.2f}B')
print('✅ Model loaded')

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA applied')

In [ ]:
from datasets import Dataset
import numpy as np

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return Dataset.from_list([json.loads(l) for l in f])

train_ds = load_jsonl('data/train.jsonl')
val_ds   = load_jsonl('data/val.jsonl')
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

def tokenise(examples):
    out = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )
    out['labels'] = out['input_ids'].copy()
    return out

remove_cols = [c for c in train_ds.column_names if c != 'text']
train_tok = train_ds.map(tokenise, batched=True, remove_columns=remove_cols, desc='Train')
val_tok   = val_ds.map(tokenise,   batched=True, remove_columns=remove_cols, desc='Val')

lens = [len(x) for x in train_tok['input_ids']]
print(f'Token lengths — min:{min(lens)} max:{max(lens)} avg:{int(np.mean(lens))}')
print('✅ Tokenisation done')

In [ ]:
from transformers import (
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling, TrainerCallback
)

# CSV loss logger
log_path = 'logs/train_log.csv'
log_file = open(log_path, 'w', newline='')
log_writer = csv.writer(log_file)
log_writer.writerow(['step', 'train_loss', 'eval_loss', 'lr', 'epoch'])

class CsvLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            log_writer.writerow([
                state.global_step,
                logs.get('loss'),
                logs.get('eval_loss'),
                logs.get('learning_rate'),
                logs.get('epoch'),
            ])
            log_file.flush()

training_args = TrainingArguments(
    output_dir='checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    weight_decay=0.01,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    fp16=False,
    bf16=False,
    dataloader_pin_memory=False,
    report_to='none',
    use_mps_device=(DEVICE == 'mps'),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[CsvLogger()],
)

print('🚀 Training started — M4 16GB estimated time: 6-10h for 3 epochs')
print('Loss logs saved to logs/train_log.csv every 10 steps')
print()

start = time.time()
trainer.train()
elapsed = int(time.time() - start)
log_file.close()

h, m = divmod(elapsed // 60, 60)
print(f'\n✅ Training done in {h}h {m}m')

In [ ]:
# Save LoRA adapter
model.save_pretrained('checkpoints/best_adapter')
tokenizer.save_pretrained('checkpoints/best_adapter')
print('✅ Adapter saved → checkpoints/best_adapter')

# Show final training loss
import pandas as pd
log_df = pd.read_csv('logs/train_log.csv').dropna(subset=['train_loss'])
print(f'\nFinal train loss : {log_df["train_loss"].iloc[-1]:.4f}')
print(f'Best eval loss   : {log_df["eval_loss"].dropna().min():.4f}')
print()
print('✅ Step 3 done — paste output in chat')

In [ ]:
# Merge LoRA weights into base model and export for Ollama
from peft import PeftModel
from transformers import AutoModelForCausalLM

print('Merging LoRA adapter into base model...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={'': 'cpu'},   # merge on CPU to avoid MPS OOM
    trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base, 'checkpoints/best_adapter')
merged = merged.merge_and_unload()
merged.save_pretrained('checkpoints/merged_model', safe_serialization=True)
tokenizer.save_pretrained('checkpoints/merged_model')
print('✅ Merged model saved → checkpoints/merged_model')

# Write Modelfile for Ollama
modelfile = """FROM ./checkpoints/merged_model
PARAMETER temperature 0
PARAMETER num_predict 200
SYSTEM \"You are SQLCoder, an expert at generating SQL queries from natural language.\"
"""
with open('Modelfile', 'w') as f:
    f.write(modelfile)

print('\n✅ Modelfile written')
print('\nTo register with Ollama, run in terminal:')
print('  ollama create sqlcoder-finetuned -f Modelfile')
print('  ollama run sqlcoder-finetuned')